# 1. A polyethylene melt from SMILES to minimized coordinates

The all-atom PhantomWalk workflow in four FlowerMD calls, followed by the
hand-off this repository adds: a periodic Sage 2.3.0 minimization at the
target density and the paper's quality metric, the energy the minimizer
removed per atom in units of Sage's largest Lennard-Jones well depth. The
structures are then exported as residue-aware PDB files (one residue per
monomer, one segment per chain) for MDAnalysis, PyMOL and NGLView.

Runs in seconds on a GPU, about a minute on a CPU.

In [ ]:
import hoomd

# GPU if one is visible, otherwise CPU. OpenMM follows the same choice.
try:
    DEVICE = hoomd.device.GPU()
    OPENMM_PLATFORM = "CUDA"
except Exception:
    DEVICE = hoomd.device.CPU()
    OPENMM_PLATFORM = "CPU"
print(DEVICE, OPENMM_PLATFORM)

In [ ]:
# The frozen all-atom PhantomWalk protocol, written out explicitly.
PROTOCOL = dict(
    bonded="uff", A=1250.0, gamma=200.0, kT=1.0, r_cut=3.5, bonded_scale=30.0,
    epsilon_weighting=True, protect_stereochemistry=True, stereo_k=30000.0,
)
RUN = dict(
    dpd_min_steps=3500, dpd_chunk=500, dpd_max_steps=40000, energy_tol=0.02,
    consecutive=2, dpd_samples_per_chunk=5, fire_steps=100, fire_dt=0.001,
    require_convergence=False,
)
DT = 0.001

In [ ]:
from pathlib import Path

# Everything this notebook writes goes here.
OUT = Path("outputs/1-polyethylene-melt")
OUT.mkdir(parents=True, exist_ok=True)

# True saves a DPD + FIRE trajectory (one GSD frame every 250 steps) for a
# movie; False writes only the starting frame.
SAVE_TRAJECTORY = True


def writers(folder):
    """Simulation keywords that put the GSD trajectory and log in `folder`."""
    folder.mkdir(parents=True, exist_ok=True)
    return dict(
        gsd_file_name=str(folder / "dpd.gsd"),
        gsd_write_freq=250 if SAVE_TRAJECTORY else 10**9,
        log_file_name=str(folder / "log.txt"),
    )

# Placement at the target density. "random walk" (the default) gives each
# chain a random orientation and position and turns every bond between two
# repeat units to a random torsion, so the chains start as overlapping random
# coils; "lattice" puts whole chains, in their built conformation, one per
# site of a grid. Both keep every bond length, bond angle and stereocenter of
# the built chains; small molecules and ions are placed as rigid units.
PLACEMENT = "random walk"


def place(molecules, density, seed):
    """Place `molecules` at `density` with the chosen PLACEMENT."""
    from flowermd.library import AllAtomLattice, AllAtomRandomWalk

    system = {"random walk": AllAtomRandomWalk, "lattice": AllAtomLattice}[PLACEMENT]
    return system(molecules, density=density, seed=seed)

In [ ]:
import unyt as u
from flowermd.library import AllAtomDPD, AllAtomPhantomWalk, PolyEthylene
from phantomwalk.all_atom import sage_handoff

chains = PolyEthylene(lengths=50, num_mols=12)          # 12 chains of 100 carbons
system = place(chains, density=0.85 * u.g / u.cm**3, seed=1)
ff = AllAtomDPD(system.system, **PROTOCOL)
sim = AllAtomPhantomWalk.from_system(system, forcefield=ff, dt=DT, device=DEVICE, seed=1,
                                     **writers(OUT))
record = sim.run_initialization(**RUN)
print(f"{record['n_particles']} atoms, DPD stationary: {record['dpd_converged']} "
      f"after {record['dpd_steps']} steps, {record['timings_s']['total']:.1f} s in total")

In [ ]:
import numpy as np

box_nm = np.asarray(ff.frame.configuration.box[:3]) / 10
handoff, minimized_a = sage_handoff(sim.to_compound(), box_nm, platform=OPENMM_PLATFORM)
print(f"Sage energy removed: {handoff['energy_removed_sage_epsilon_atom']:.2f} eps_max per atom "
      f"(finite: {handoff['finite']}, minimizer {handoff['minimizer_s']:.1f} s)")
sim.write_record(OUT / "record.json")

`record` holds every setting, the wall time of each phase, the DPD energy
history and the software versions; it is saved as `record.json`. PE chain
ends are residue `PET`, the other monomers `PEM`.

## Export the structures

`residue_topology` reads the FlowerMD hierarchy (melt, molecule, monomer)
into PDB labels: one residue per monomer, one segment ID per molecule (four
base-36 characters, up to 1.68 million molecules), CONECT records for every
bond and the box in CRYST1. Molecules are written whole with their centroids
in the box, so per-chain quantities such as Rg work directly. Three stages
are written:

| file | coordinates |
|---|---|
| `placement.pdb` | the placement before DPD |
| `initialized.pdb` | after DPD + FIRE, what `sim.to_compound()` hands off |
| `minimized.pdb` | after the Sage 2.3.0 minimization; **use this one for analysis** |

`dpd.dcd` is the DPD + FIRE trajectory (whole, continuous molecules) with
`initialized.pdb` as its topology; `movie.pml` and `minimized.pml` open them
in PyMOL (`cd` into the output folder, then `pymol movie.pml`).

In [ ]:
from phantomwalk.all_atom import (
    flush_trajectory, frame_positions, residue_topology, write_pdb, write_trajectory,
)
from phantomwalk.all_atom.visualization import write_pymol_script


def export(folder, sim, ff, minimized_a):
    """Write the three stages, the DPD trajectory and the PyMOL scripts."""
    top = residue_topology(sim.system.system, box_a=ff.frame.configuration.box[:3])
    initialized_a = sim.final_positions() * 10
    write_pdb(top, frame_positions(ff.frame), folder / "placement.pdb")
    write_pdb(top, initialized_a, folder / "initialized.pdb")
    write_pdb(top, minimized_a, folder / "minimized.pdb")
    flush_trajectory(sim)
    # FIRE's last steps rarely land on the GSD period; append the final frame
    write_trajectory(top, folder / "dpd.gsd", folder / "dpd.dcd",
                     append_positions_a=initialized_a)
    write_pymol_script(folder / "initialized.pdb", folder / "movie.pml",
                       trajectory=folder / "dpd.dcd")
    write_pymol_script(folder / "minimized.pdb", folder / "minimized.pml")
    return top

top = export(OUT, sim, ff, minimized_a)
print(f"{top.n_molecules} molecules, {len(set(zip(top.molecule, top.resids)))} residues, "
      f"{len(top.bonds)} bonds -> {sorted(p.name for p in OUT.iterdir())}")

## Look at it

NGLView draws every atom as licorice and keeps only the CONECT bonds (NGL
would otherwise guess bonds between atoms that overlap during DPD).
`color="chainname"` gives one color per molecule, `"resname"` one per residue
name, `"element"` the usual element colors. The movie needs
`SAVE_TRAJECTORY = True`; `save_gif(movie, n_frames, "dpd.gif")` records it
from a live notebook.

In [ ]:
from phantomwalk.all_atom.visualization import save_gif, show_movie, show_structure

show_structure(OUT / "minimized.pdb")

In [ ]:
movie = show_movie(OUT / "initialized.pdb", OUT / "dpd.dcd") if SAVE_TRAJECTORY else None
movie

## Analysis starts from an MDAnalysis Universe

Residues are monomers and segments are chains, and bonds come from CONECT,
so `u.segments` iterates chains. Notebook 4 goes further (Rg, end-to-end
distance, structure factor) for the Colina benchmark polymers.

In [ ]:
from phantomwalk.all_atom import universe

u_min = universe(OUT / "minimized.pdb")
rg = np.array([chain.atoms.radius_of_gyration() for chain in u_min.segments])
print(f"{u_min.segments.n_segments} chains, {u_min.residues.n_residues} residues; "
      f"Rg = {rg.mean():.1f} +/- {rg.std():.1f} A")